# 02 OIFS: the Eocene atmosphere and land surface

The `oifs` step (`EoceneOIFS`) rewrites the OpenIFS initial and climate files with the Eocene geography of
Herold et al. (2014):

* `ICMGGECE4INIT` (surface, grid point): land-sea mask and its gradient, sub-grid orography (`sdor`, `slor`,
  `isor`), vegetation types and cover (`tvh`, `tvl`, `cvh`, `cvl`), soil type, surface albedos; snow, lakes and a
  few other fields set to zero;
* `ICMSHECE4INIT` (spectral): orography (`z`) from Herold; the other spectral fields truncated to their global mean;
* `ICMGGECE4INIUA` (upper air): humidity and clouds set to zero;
* `ICMCLECE4` (monthly climate): albedos and leaf area index;
* `bare_soil_albedos.grb`: bare soil albedos.

**Run `01_quickstart.ipynb` first**: this notebook reads its configuration and output. Each figure shows the
present-day input (left) and the Eocene output (right) with the same colour scale.

In [ ]:
import os
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import xarray as xr

from ecefast.core.config import load_config
from ecefast.core.workspace import Context

# Working directory of the tutorials: 01_quickstart.ipynb writes the config and the outputs here
WORK = Path(os.environ.get("ECEFAST_TUTORIAL_DIR", Path(tempfile.gettempdir()) / "ecefast-tutorial"))
CONFIG = WORK / "laptop.yml"

ctx = Context(load_config(str(CONFIG)))  # expands path templates such as "{input}/oasis/{nemo_grid}/rstos.nc"

from ecefast.paleo.eocene.oifs.eoceneOIFS import EoceneOIFS
from ecefast.paleo.eocene.plots.helper_functions import load_grib_var_spec, plot_before_after

## What the step reads and writes

In [ ]:
for kind in ("needs", "reads", "writes"):
    print(f"{kind}:")
    for template in getattr(EoceneOIFS, kind):
        print("   ", ctx.expand(template) if kind != "needs" else template)

## The Herold et al. (2014) inputs

Eocene topography (m; negative values are ocean depth) and the Eocene biomes, from which the vegetation is
derived:

In [ ]:
herold = Path(ctx.paths.herold)
topo = xr.open_dataset(herold / "herold_etal_eocene_topo_1x1.nc")["topo"]
biome = xr.open_dataset(herold / "herold_etal_eocene_biome_1x1.nc")["eocene_biome"]
fig, axes = plt.subplots(1, 2, figsize=(13, 4), layout="constrained")
topo.plot(ax=axes[0], cmap="terrain", vmin=-6000, vmax=6000)
biome.plot(ax=axes[1], cmap="tab20")
axes[0].set_title("Eocene topography")
axes[1].set_title("Eocene biome")

## Before and after

`load_grib_var_spec` reads one GRIB variable with CDO: grid-point fields are remapped to a 1° grid
(nearest neighbour), spectral fields are transformed to their Gaussian grid. `compare` loads the same file from the
input and output trees.

In [ ]:
def compare(template, var, title, **kwargs):
    """Plot `var` of the file `template` (relative to the tree root) from the input and the output tree."""
    before = load_grib_var_spec(ctx.expand("{input}/" + template), var)
    after = load_grib_var_spec(ctx.expand("{output}/" + template), var)
    return plot_before_after(before, after, title, **kwargs)


INIT = "oifs/{oifs_grid}/19900101/ICMGGECE4INIT"
SPECTRAL = "oifs/{oifs_grid}/19900101/ICMSHECE4INIT"
CLIMATE = "oifs/{oifs_grid}/climate.v020/ICMCLECE4"
BARE_SOIL = "oifs/{oifs_grid}/climate.v020/bare_soil_albedos.grb"

### Land-sea mask (`lsm`)

In [ ]:
compare(INIT, "lsm", "Land-sea mask", cmap="Greens");

### Orography

The orography is the surface geopotential `z` of the spectral file, divided by g = 9.81 m s⁻² (the value used by
the step) to show metres.

In [ ]:
G = 9.81  # m s-2, as in EoceneOIFS.create_sh
units = {"long_name": "orography", "units": "m"}
before = (load_grib_var_spec(ctx.expand("{input}/" + SPECTRAL), "z") / G).assign_attrs(units)
after = (load_grib_var_spec(ctx.expand("{output}/" + SPECTRAL), "z") / G).assign_attrs(units)
plot_before_after(before, after, "Orography (m)", cmap="terrain");

### Sub-grid orography: standard deviation (`sdor`) and slope (`slor`)

In [ ]:
compare(INIT, "sdor", "Standard deviation of sub-grid orography (m)", cmap="viridis")
compare(INIT, "slor", "Slope of sub-grid orography", cmap="viridis");

### Vegetation

High (`tvh`) and low (`tvl`) vegetation types (IFS classes; 0 = none) and their cover (`cvh`, `cvl`), from the
Herold biomes.

In [ ]:
compare(INIT, "tvh", "Type of high vegetation", cmap="tab20")
compare(INIT, "tvl", "Type of low vegetation", cmap="tab20")
compare(INIT, "cvh", "High vegetation cover", cmap="YlGn")
compare(INIT, "cvl", "Low vegetation cover", cmap="YlGn");

### Albedo: initial file (`al`) and monthly climate file (`ICMCLECE4`, January and July)

In [ ]:
compare(INIT, "al", "Albedo, ICMGGECE4INIT", cmap="cividis")

for month, name in [(0, "January"), (6, "July")]:
    before = load_grib_var_spec(ctx.expand("{input}/" + CLIMATE), "al").isel(time=month)
    after = load_grib_var_spec(ctx.expand("{output}/" + CLIMATE), "al").isel(time=month)
    plot_before_after(before, after, f"Albedo, ICMCLECE4, {name}", cmap="cividis")

### Leaf area index of high vegetation (`lai_hv`, ICMCLECE4, July)

In [ ]:
before = load_grib_var_spec(ctx.expand("{input}/" + CLIMATE), "lai_hv").isel(time=6)
after = load_grib_var_spec(ctx.expand("{output}/" + CLIMATE), "lai_hv").isel(time=6)
plot_before_after(before, after, "Leaf area index, high vegetation, July", cmap="YlGn");

### Bare soil albedo (`bare_soil_albedos.grb`)

The four variables of this file have no short name in the GRIB tables CDO uses; `var117` is the first one.

In [ ]:
compare(BARE_SOIL, "var117", "Bare soil albedo (var117)", cmap="cividis");

### Upper air

`ICMGGECE4INIUA` keeps temperature and ozone; humidity and clouds (`q`, `clwc`, `ciwc`, `crwc`, `cswc`, `cc`) are
set to zero, so the model starts from a dry atmosphere. The aerosol climatology (`eocene.oifs.aerosols`) is off
by default and not shown here.